In [1]:
import pandas as pd
import numpy as np

class BarcodeProcessor:
    def __init__(self, lookup_table_path,data_drop_table_path):
        self.lookup_table = self.load_lookup_table(lookup_table_path) 
        self.filter_lookup_table(data_drop_table_path) # lookup_table filtering
        #self.lookup_table.to_csv("indexing/ALT_REF_LookUpTable_filtered_20231111.csv",index=None)

    def load_lookup_table(self, file_path):
        """
        Load lookup table from CSV file.
        """
        return pd.read_csv(file_path,header=None)

    def filter_lookup_table(self, file_path):
        """
        Filter lookup table based on criteria defined in another CSV.
        """
        df_to_drop = pd.read_csv(file_path, index_col=0)
        strings_to_drop = df_to_drop[df_to_drop["Drop"] == "Y"].index
        for string_to_drop in strings_to_drop:
            self.lookup_table = self.lookup_table[~self.lookup_table.apply(lambda row: row.astype(str).str.contains(string_to_drop).any(), axis=1)]


    def process_columns(self, count_table):
        """
        Process columns to separate and calculate 'REF' and 'ALT' data.
        """
        count_table_new = pd.DataFrame()

        # Exclude 'seq_id' column explicitly
        columns_to_process = [col for col in count_table.columns if col != 'seq_id']

        for col in columns_to_process:
            df_tmp = count_table[[col, "seq_id"]]
            alt = df_tmp.loc[self.lookup_table[0]].set_index("seq_id")
            ref = df_tmp.loc[self.lookup_table[1]].set_index("seq_id")
            count_table_new[col + "_ALT"] = alt[col]
            count_table_new[col + "_REF"] = ref[col].tolist()
        return count_table_new

    def reshape_group(self, group, samples, max_barcodes):
        """
        Reshape the grouped data for each enhancer.
        """
        n_samples = len(samples)
        n_barcodes = group.shape[0]  # Number of rows in the group
        reshaped_data = {}
        
        for i in range(n_samples):
            for j in range(max_barcodes):
                column_name = f'{samples[i]}_Barcode_{j+1}'
                reshaped_data[column_name] = group.iloc[j, i + 1] if j < n_barcodes else np.nan
        return pd.Series(reshaped_data)

    def convert_REFALT_Barcodes(self, count_table_path, experiment_name, nucleic_acid_type):
        """
        Convert REF/ALT barcodes for a given nucleic acid type and experiment.
        """
        # Construct file paths
        lookup_file_path = f"read_counts_R1R2/{experiment_name}_{nucleic_acid_type}_matched_barcodes.csv"
        output_file_path = f"read_counts_R1R2/{experiment_name}_{nucleic_acid_type}_matched_barcodes_reshaped.csv"

        # Load count table and set index
        count_table = pd.read_csv(count_table_path, index_col=0)
        count_table['seq_id'] = count_table.index
        count_table.set_index('enhancer_id', inplace=True)

        # Process columns
        processed_count_table = self.process_columns(count_table)

        # Load lookup table, rename axis, and merge with processed count table
        lookup_table = pd.read_csv(lookup_file_path, index_col=0)['enhancer_id']
        lookup_table = lookup_table.rename_axis('seq_id').reset_index()
        annotated_df = processed_count_table.merge(lookup_table, on="seq_id", how='left')

        # Reshape data
        max_barcodes = annotated_df.groupby('enhancer_id').size().max()
        columns_to_keep = [col for col in annotated_df.columns if col not in ['seq_id', 'enhancer_id']]
        reshaped_df = annotated_df.groupby('enhancer_id').apply(lambda group: self.reshape_group(group, columns_to_keep, max_barcodes))

        # Save reshaped dataframe
        reshaped_df.fillna(0).to_csv(output_file_path)
        return reshaped_df


# Example usage of the class
# Initialize the BarcodeProcessor
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_20231117.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")
#processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_filtered_amended_motifOnly_20240605.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")

In [1]:
import pandas as pd
import numpy as np

class BarcodeProcessor:
    def __init__(self, lookup_table_path, data_drop_table_path):
        self.lookup_table = self.load_lookup_table(lookup_table_path)
        self.filter_lookup_table(data_drop_table_path)

        # 强制成 string，避免 "00123" 这种被当成数字导致匹配失败
        self.lookup_table[0] = self.lookup_table[0].astype(str)
        self.lookup_table[1] = self.lookup_table[1].astype(str)

    def load_lookup_table(self, file_path):
        return pd.read_csv(file_path, header=None)

    def filter_lookup_table(self, file_path):
        df_to_drop = pd.read_csv(file_path, index_col=0)
        strings_to_drop = df_to_drop.loc[df_to_drop["Drop"] == "Y"].index.astype(str)

        for s in strings_to_drop:
            self.lookup_table = self.lookup_table[
                ~self.lookup_table.apply(lambda row: row.astype(str).str.contains(s).any(), axis=1)
            ]

    def process_columns(self, count_table):
        """
        输入 count_table:
          - index = seq_id (barcode)
          - columns = samples...
        输出 processed_count_table:
          - index = ALT seq_id（与 lookup_table[0] 顺序一致）
          - columns = 每个 sample 的 *_ALT / *_REF
        """
        # 只处理样本列（排除 seq_id / enhancer_id 之类注释列）
        columns_to_process = [c for c in count_table.columns if c not in ("seq_id", "enhancer_id")]

        alt_ids = self.lookup_table[0].to_numpy()
        ref_ids = self.lookup_table[1].to_numpy()

        # 用 reindex 更稳：如果有缺失会变 NaN（比 loc 直接 KeyError 更好 debug）
        alt_block = count_table.reindex(alt_ids)
        ref_block = count_table.reindex(ref_ids)

        out = pd.DataFrame(index=alt_ids)
        out.index.name = "seq_id"

        for col in columns_to_process:
            out[f"{col}_ALT"] = alt_block[col].to_numpy()
            out[f"{col}_REF"] = ref_block[col].to_numpy()

        return out

    def reshape_group(self, group, sample_cols, max_barcodes):
        """
        group: 一个 enhancer_id 下的多行（每行是一个 barcode/seq_id）
        sample_cols: 要 reshape 的列（例如所有 *_ALT / *_REF）
        """
        reshaped = {}
        for col in sample_cols:
            vals = group[col].to_numpy()
            if len(vals) < max_barcodes:
                vals = np.pad(vals, (0, max_barcodes - len(vals)), constant_values=np.nan)
            else:
                vals = vals[:max_barcodes]

            for j in range(max_barcodes):
                reshaped[f"{col}_Barcode_{j+1}"] = vals[j]

        return pd.Series(reshaped)

    def convert_REFALT_Barcodes(self, count_table_path, experiment_name, nucleic_acid_type):
        """
        关键点：
          - count_table_path: 读入计数（index=seq_id）
          - enhancer_id mapping: 从 lookup_file_path 读入 (index=seq_id, column=enhancer_id)
        """
        lookup_file_path = f"read_counts_R1R2/{experiment_name}_{nucleic_acid_type}_matched_barcodes.csv"
        output_file_path = f"read_counts_R1R2/{experiment_name}_{nucleic_acid_type}_matched_barcodes_reshaped.csv"

        # 1) 读 count table：保持 index=seq_id
        count_table = pd.read_csv(count_table_path, index_col=0)
        count_table.index = count_table.index.astype(str)
        count_table.index.name = "seq_id"
        count_table["seq_id"] = count_table.index  # 只是为了调试/兼容；后面主要用 index

        # 2) 生成 ALT/REF paired 矩阵（index=ALT seq_id）
        processed = self.process_columns(count_table)  # index=seq_id(ALT)

        # 3) 读 seq_id -> enhancer_id 注释表
        anno = pd.read_csv(lookup_file_path, index_col=0)
        anno.index = anno.index.astype(str)
        if "enhancer_id" not in anno.columns:
            raise KeyError(f"`enhancer_id` not found in {lookup_file_path}. columns={anno.columns.tolist()}")

        # 4) 合并注释（用 index join 最稳）
        annotated = processed.join(anno["enhancer_id"], how="left")

        n_missing = annotated["enhancer_id"].isna().sum()
        if n_missing > 0:
            ex = annotated.index[annotated["enhancer_id"].isna()][:5].tolist()
            raise ValueError(
                f"{n_missing} ALT seq_id 没有匹配到 enhancer_id（检查 lookup_file_path / seq_id 类型）。例子: {ex}"
            )

        # 5) reshape：每个 enhancer_id 展开成固定条码数
        sample_cols = [c for c in annotated.columns if c != "enhancer_id"]
        max_barcodes = annotated.groupby("enhancer_id").size().max()

        reshaped_df = annotated.groupby("enhancer_id", sort=False).apply(
            lambda g: self.reshape_group(g, sample_cols, max_barcodes)
        )

        reshaped_df.fillna(0).to_csv(output_file_path)
        return reshaped_df


In [ ]:
#202404012 THP1 mono pseudobarcode
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_pseudobarcode_20240404.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/THP1mono20240412_RNA_matched_barcodes.csv", experiment_name="THP1mono20240412", nucleic_acid_type="RNA")

In [ ]:
#202404012 THP1 mono pseudobarcode
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_pseudobarcode_20240404.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/THP1mono20240412_RNA_matched_barcodes.csv", experiment_name="THP1mono20240412", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/THP1mono20240412_DNA_matched_barcodes.csv", experiment_name="THP1mono20240412", nucleic_acid_type="DNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/THP1mono20240412Pseudo_DNA_matched_barcodes.csv", experiment_name="THP1mono20240412Pseudo", nucleic_acid_type="DNA")


In [ ]:
#202404017 THP1 mac pseudobarcode
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_pseudobarcode_20240404.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/THP1MacrophagePseudobarcodes_RNA_matched_barcodes.csv", experiment_name="THP1MacrophagePseudobarcodes", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/THP1MacrophagePseudobarcodes_DNA_matched_barcodes.csv", experiment_name="THP1MacrophagePseudobarcodes", nucleic_acid_type="DNA")


In [ ]:
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_pseudobarcode_20240404.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")

reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged20240404BrainSum_DNA_matched_barcodes.csv", experiment_name="BrainR1R2merged20240404BrainSum", nucleic_acid_type="DNA")

In [ ]:
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_pseudobarcode_20240404.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/GutR1R2merged20240404_RNA_matched_barcodes.csv", experiment_name="GutR1R2merged20240404", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/GutR1R2merged20240404_DNA_matched_barcodes.csv", experiment_name="GutR1R2merged20240404", nucleic_acid_type="DNA")

In [ ]:
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/HEK293T_RNA_matched_barcodes.csv", experiment_name="HEK293T", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/HEK293T_DNA_matched_barcodes.csv", experiment_name="HEK293T", nucleic_acid_type="DNA")

In [ ]:
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/Brain_RNA_matched_barcodes.csv", experiment_name="Brain", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/Brain_DNA_matched_barcodes.csv", experiment_name="Brain", nucleic_acid_type="DNA")

In [ ]:
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged_RNA_matched_barcodes.csv", experiment_name="BrainR1R2merged", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged_DNA_matched_barcodes.csv", experiment_name="BrainR1R2merged", nucleic_acid_type="DNA")

In [ ]:
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged20240326_RNA_matched_barcodes.csv", experiment_name="BrainR1R2merged20240326", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged20240326_DNA_matched_barcodes.csv", experiment_name="BrainR1R2merged20240326", nucleic_acid_type="DNA")

In [ ]:
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/GutR1R2merged20240326_RNA_matched_barcodes.csv", experiment_name="GutR1R2merged20240326", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/GutR1R2merged20240326_DNA_matched_barcodes.csv", experiment_name="GutR1R2merged20240326", nucleic_acid_type="DNA")

In [ ]:
processor = BarcodeProcessor(lookup_table_path="indexing/ALT_REF_LookUpTable_amended_pseudobarcode_20240404.csv",data_drop_table_path = "indexing/low_frequency_enhancer_drop_table_20231111.csv")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged20260121_RNA_matched_barcodes_enhancerid.csv", experiment_name="BrainR1R2merged20260121", nucleic_acid_type="RNA")
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_enhancerid.csv", experiment_name="BrainR1R2merged20260121", nucleic_acid_type="DNA")

KeyError: "None of ['enhancer_id'] are in the columns"

In [3]:
reshaped_df = processor.convert_REFALT_Barcodes(count_table_path="read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_enhancerid.csv", experiment_name="BrainR1R2merged20260121", nucleic_acid_type="DNA")

KeyError: 'enhancer_id'

In [ ]:
import pandas as pd
import numpy as np

pool_path = "read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_pool.csv"
out_path  = "read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_withcontrol_from_pool.csv"

df = pd.read_csv(pool_path, index_col=0)

# 样本列：除 enhancer_id 之外的所有列
sample_cols = [c for c in df.columns if c != "enhancer_id"]

# 每个 enhancer 保留多少个 barcode（你的 withcontrol 示例是 5）
max_barcodes = 5

# 保留原始文件顺序：每个 enhancer 按出现顺序取前 N 个 barcode
df["_row_order"] = np.arange(len(df), dtype=int)
df = df.sort_values(["enhancer_id", "_row_order"])

# enhancer 内部 barcode 编号：1..N
df["_bc_rank"] = df.groupby("enhancer_id").cumcount() + 1
df = df[df["_bc_rank"] <= max_barcodes].copy()

# 变宽：每个样本 -> 样本_ALT_Barcode_i
wide_blocks = []
for col in sample_cols:
    tmp = df.pivot(index="enhancer_id", columns="_bc_rank", values=col)
    tmp.columns = [f"{col}_ALT_Barcode_{i}" for i in tmp.columns]
    wide_blocks.append(tmp)

out = pd.concat(wide_blocks, axis=1).fillna(0).sort_index()
out.to_csv(out_path)

print(out.shape, out_path)


(1631, 60) read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_withcontrol_from_pool.csv


In [5]:
import pandas as pd
import numpy as np
import re
from collections import defaultdict

pool_path = "read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_pool.csv"
target_path = "read_counts_R1R2/BrainR1R2merged20260121_RNA_matched_barcodes_reshaped_altref_withcontrol.csv"
out_path = "read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_pool_withcontrol.csv"

pool = pd.read_csv(pool_path, index_col=0)
target = pd.read_csv(target_path, index_col=0)

# Identify target sample base names and max barcode count per base name
# Expected column pattern: "<sample>_ALT_Barcode_<n>" or "<sample>_REF_Barcode_<n>"
pat = re.compile(r"^(.*)_(ALT|REF)_Barcode_(\d+)$")
base_to_n = defaultdict(int)
for c in target.columns:
    m = pat.match(c)
    if m:
        base = m.group(1)
        n = int(m.group(3))
        base_to_n[base] = max(base_to_n[base], n)

# Fallback: if target columns don't match pattern, use a global default
if not base_to_n:
    raise ValueError("Could not infer barcode counts from target column names. Please check target format.")

# Pool sample columns (excluding enhancer_id)
pool_sample_cols = [c for c in pool.columns if c != "enhancer_id"]

# For each base in target, we will build ALT barcodes from pool column with the same base name (if exists)
# If pool doesn't have that column, fill zeros.
# We keep enhancer order as in target (index)
pool["_row_order"] = np.arange(len(pool), dtype=int)
pool = pool.sort_values(["enhancer_id", "_row_order"])
pool["_bc_rank"] = pool.groupby("enhancer_id").cumcount() + 1

wide_blocks = []
for base, max_n in base_to_n.items():
    if base in pool_sample_cols:
        tmp = pool.loc[pool["_bc_rank"] <= max_n].pivot(index="enhancer_id", columns="_bc_rank", values=base)
        tmp = tmp.reindex(target.index)  # match enhancers and order
        tmp.columns = [f"{base}_ALT_Barcode_{i}" for i in tmp.columns]
    else:
        # create empty block
        tmp = pd.DataFrame(index=target.index, columns=[f"{base}_ALT_Barcode_{i}" for i in range(1, max_n+1)], dtype=float)
    wide_blocks.append(tmp)

dna_wide_alt = pd.concat(wide_blocks, axis=1).fillna(0)

# Now add REF columns if they exist in target: set to 0 unless you have ref counts elsewhere
# (Pool table typically contains ALT only. If you have REF too, adapt here.)
ref_cols = [c for c in target.columns if "_REF_Barcode_" in c]
if ref_cols:
    dna_wide_ref = pd.DataFrame(0, index=target.index, columns=ref_cols, dtype=float)
    out = pd.concat([dna_wide_alt, dna_wide_ref], axis=1)
else:
    out = dna_wide_alt

# Match exact target column order where possible
common_cols = [c for c in target.columns if c in out.columns]
missing_in_out = [c for c in target.columns if c not in out.columns]
# Add any missing columns as zeros to fully match format
for c in missing_in_out:
    out[c] = 0.0
out = out[target.columns]  # exact order

out.to_csv(out_path)

(out.shape, out_path, len(missing_in_out), missing_in_out[:5])



((1610, 60),
 'read_counts_R1R2/BrainR1R2merged20260121_DNA_matched_barcodes_pool_withcontrol.csv',
 0,
 [])